# Good Provocation in Arthouse — Deep Dive
## Companion to `general_analysis_executed.ipynb` · DTU Spring 2026

**One thesis:** in arthouse, *formal* provocation outperforms *transgressive* provocation. The audience rewards how a film is told, not what's in it.

Three visuals defend that claim:
1. **Headline scatter** — every cohort film placed on (polarization × mean rating). Two distinct controversy regimes appear.
2. **Mechanism** — paired rating distributions showing two films can have the same polarization score and opposite audience meaning.
3. **Provocation lexicon** — keywords ranked by their productive-provocation lift, turning the thesis into a shortlist for greenlighting and positioning.

Everything builds on the polarization metric defined in §2 of the general analysis.

In [ ]:
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display, Markdown

# Locate the project root by walking up until we see the canonical layout
PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / '03-data').exists() and (candidate / 'src').exists():
        PROJECT_ROOT = candidate
        break
os.chdir(PROJECT_ROOT)

COHORT_PATH = PROJECT_ROOT / 'notebooks' / 'arthouse' / 'arthouse-analysis' / 'arthouse_cohort.csv'
cohort = pd.read_csv(COHORT_PATH)

# Palette — keep continuity with general_analysis, add green/red for the thesis
PRIMARY = '#3a5a78'   # cohort baseline
ACCENT  = '#c46d4a'
GRAY    = '#888888'
PRODUCT = '#2e8b57'   # productive provocation
REJECT  = '#a63333'   # rejected provocation

# Figures land here, ready to drop into slides
FIG_DIR = PROJECT_ROOT / 'notebooks' / 'arthouse' / 'arthouse-analysis' / 'figures' / 'provocation'
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.size': 10.5,
    'axes.titlesize': 12,
    'axes.labelsize': 10.5,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

print(f'Cohort films: {len(cohort):,}')
print(f'Loaded from:  {COHORT_PATH}')
print(f'Figures →     {FIG_DIR}')

## §1 — Polarization is a trap

The polarization score from the general analysis (share of votes in 1–2 + 9–10) tells you *that* a film divided audiences. It cannot tell you *how*.

Two films with identical polar = 0.40 might mean very different things:

- One has 35% of votes in 9–10 and 5% in 1–2 — passionately admired by most, dismissed by a small refusing minority. Mean rating is high.
- The other has 5% in 9–10 and 35% in 1–2 — rejected by most, defended by a small advocating minority. Mean rating is low.

Same controversy score. Opposite audience truth. The fix: split polarization into a **love share** and a **hate share**, and read it alongside the mean rating computed from the distribution.

In [ ]:
mask = cohort['rating_1'].notna() & (cohort['totalVotes'].fillna(0) >= 100)
films = cohort.loc[mask].copy()

tail_high = films['rating_9']  + films['rating_10']
tail_low  = films['rating_1']  + films['rating_2']
films['polar']       = (tail_high + tail_low) / films['totalVotes']
films['love_share']  = tail_high / films['totalVotes']
films['hate_share']  = tail_low  / films['totalVotes']
films['mean_rating'] = sum(i * films[f'rating_{i}'] for i in range(1, 11)) / films['totalVotes']

summary = pd.DataFrame({
    'cohort mean': [
        round(films['polar'].mean(), 3),
        round(films['love_share'].mean(), 3),
        round(films['hate_share'].mean(), 3),
        round(films['mean_rating'].mean(), 2),
    ]
}, index=['polarization', 'love share (9–10)', 'hate share (1–2)', 'mean rating'])
display(Markdown(f'**Decomposed polarization on {len(films):,} films with ≥100 votes**'))
display(summary)

## §2 — The headline: two kinds of arthouse controversy

We split the usable films into four quadrants on (polarization × mean rating), with thresholds at the cohort medians.

- **Productive provocation** (top-right) — high polar + high rating. Controversy that converts.
- **Rejected provocation** (bottom-right) — high polar + low rating. Controversy that fails.
- **Critical consensus** (top-left) — low polar + high rating. Quiet wins.
- **Forgotten** (bottom-left) — low polar + low rating. The long tail.

The thesis lives in the contrast between the two right-hand quadrants. Same controversy magnitude, opposite verdict.

In [ ]:
polar_med  = films['polar'].median()
rating_med = films['mean_rating'].median()

def quadrant(row):
    if row['polar'] >= polar_med and row['mean_rating'] >= rating_med: return 'productive'
    if row['polar'] >= polar_med and row['mean_rating'] <  rating_med: return 'rejected'
    if row['polar'] <  polar_med and row['mean_rating'] >= rating_med: return 'consensus'
    return 'forgotten'

films['quadrant'] = films.apply(quadrant, axis=1)

# Quadrant counts
rows = []
for q in ['productive', 'rejected', 'consensus', 'forgotten']:
    sub = films[films['quadrant'] == q]
    rows.append({'quadrant': q, 'n': len(sub),
                 'mean polar': round(sub['polar'].mean(), 3),
                 'mean rating': round(sub['mean_rating'].mean(), 2)})
display(Markdown('**Films per quadrant**'))
display(pd.DataFrame(rows).set_index('quadrant'))

# Anchor films — pick the highest-vote (= most recognizable) titles in each quadrant
def pick_anchors(quad, n=4, min_votes=20_000):
    sub = films[(films['quadrant'] == quad) & (films['totalVotes'] >= min_votes)]
    return sub.sort_values('totalVotes', ascending=False).head(n)

anchors = {q: pick_anchors(q) for q in ['productive', 'rejected', 'consensus']}

# --- Plot ---
fig, ax = plt.subplots(figsize=(11, 6.8))
qcolors = {'productive': PRODUCT, 'rejected': REJECT,
           'consensus':  PRIMARY,  'forgotten': GRAY}

for q, c in qcolors.items():
    sub = films[films['quadrant'] == q]
    ax.scatter(sub['polar'], sub['mean_rating'],
               s=15, alpha=0.45, color=c,
               label=f'{q.capitalize()}  (n={len(sub):,})',
               edgecolor='none', zorder=2)

# Set limits before adding shading
ax.set_xlim(films['polar'].min() - 0.02, films['polar'].max() + 0.02)
ax.set_ylim(films['mean_rating'].min() - 0.3, films['mean_rating'].max() + 0.3)
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()

# Quadrant shading — subtle
ax.add_patch(Rectangle((polar_med, rating_med), xmax-polar_med, ymax-rating_med,
                       facecolor=PRODUCT, alpha=0.07, zorder=0))
ax.add_patch(Rectangle((polar_med, ymin), xmax-polar_med, rating_med-ymin,
                       facecolor=REJECT, alpha=0.07, zorder=0))

ax.axvline(polar_med,  color='black', lw=0.7, ls='--', alpha=0.5)
ax.axhline(rating_med, color='black', lw=0.7, ls='--', alpha=0.5)

# Annotate anchor films
for q, sub in anchors.items():
    for _, row in sub.iterrows():
        title = row.get('englishTitle') if pd.notna(row.get('englishTitle')) else row.get('originalTitle')
        if pd.isna(title): continue
        ax.annotate(str(title)[:30],
                    xy=(row['polar'], row['mean_rating']),
                    xytext=(6, 4), textcoords='offset points',
                    fontsize=8.5, color=qcolors[q],
                    fontweight='medium')

# Quadrant labels in the corners
ax.text(xmax - 0.005, ymax - 0.05, 'PRODUCTIVE\nPROVOCATION',
        ha='right', va='top', fontsize=11, fontweight='bold',
        color=PRODUCT, alpha=0.7)
ax.text(xmax - 0.005, ymin + 0.05, 'REJECTED\nPROVOCATION',
        ha='right', va='bottom', fontsize=11, fontweight='bold',
        color=REJECT, alpha=0.7)

ax.set_xlabel('Polarization  (share of votes in 1–2 + 9–10)')
ax.set_ylabel('Mean IMDb rating  (computed from vote distribution)')
ax.set_title(f'Two kinds of arthouse controversy · {len(films):,} cohort films with ≥100 votes',
             pad=12)
ax.legend(frameon=False, loc='lower left', fontsize=9.5)
for s in ('top','right'): ax.spines[s].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / '01_headline_scatter.png', dpi=200, bbox_inches='tight')
plt.show()

**Reading the chart:**
- The right side of the plot is the entire "polarization story." Films above the polarization median split sharply on the rating axis.
- Productive (green) holds the canonical arthouse — passionate love + a refusing minority.
- Rejected (red) holds the films audiences walked out of — controversy that did not convert.
- The mean-rating axis is exactly the second axis the polarization score alone cannot give you.

## §3 — Mechanism: same polarization, opposite shape

To make the trap visible, line up two films with similar polarization scores but opposite mean ratings, plus a consensus film for contrast. The shapes of their vote distributions are the actual story.

In [ ]:
# Restrict to ≥5,000 votes for a clean visual shape
big = films[films['totalVotes'] >= 5_000].copy()

prod = big[big['quadrant'] == 'productive'].nlargest(1, 'polar').iloc[0]
rej  = big[big['quadrant'] == 'rejected'].nlargest(1, 'polar').iloc[0]
cons = big[big['quadrant'] == 'consensus'].nlargest(1, 'mean_rating').iloc[0]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
specs = [(prod, 'Productive provocation', PRODUCT),
         (rej,  'Rejected provocation',   REJECT),
         (cons, 'Critical consensus',     PRIMARY)]

for ax, (row, label, color) in zip(axes, specs):
    counts = [row[f'rating_{i}'] for i in range(1, 11)]
    ax.bar(range(1, 11), counts, color=color, alpha=0.85)
    title = row.get('englishTitle') if pd.notna(row.get('englishTitle')) else row.get('originalTitle')
    title = str(title)[:34] if pd.notna(title) else '?'
    yr = int(row['releaseYear']) if pd.notna(row['releaseYear']) else '?'
    ax.set_title(f'{label}\n{title} ({yr})\n'
                 f'polar={row["polar"]:.2f}  ·  rating={row["mean_rating"]:.1f}',
                 fontsize=10.5)
    ax.set_xticks(range(1, 11))
    ax.set_xlabel('IMDb rating')
    for s in ('top','right'): ax.spines[s].set_visible(False)
axes[0].set_ylabel('Vote count')

plt.tight_layout()
plt.savefig(FIG_DIR / '02_mechanism_distributions.png', dpi=200, bbox_inches='tight')
plt.show()

print(f'Productive pick: {prod.get("englishTitle") or prod.get("originalTitle")}  (polar {prod["polar"]:.2f})')
print(f'Rejected pick:   {rej.get("englishTitle") or rej.get("originalTitle")}  (polar {rej["polar"]:.2f})')
print(f'Consensus pick:  {cons.get("englishTitle") or cons.get("originalTitle")}  (polar {cons["polar"]:.2f})')

**Reading the shapes:**
- Productive provocation — heavy mass at 9–10 with a smaller refusing tail at 1–2. A vocal minority cannot kill it.
- Rejected provocation — heavy mass at 1–2 with a smaller defending tail at 9–10. A loyal niche cannot save it.
- Consensus — unimodal, no tails. Beloved without controversy.

The shape *is* the audience verdict. A single polarization score collapses these into the same number.

## §4 — The provocation lexicon

If the thesis is right — formal provocation works, transgressive provocation does not — it should show up at the keyword level. Films sharing a productive-provocation theme should sit higher on the rating axis, on average, than films sharing a rejected-provocation theme, even though both have above-cohort polarization.

**Method:** explode every film's IMDb keywords, drop title-pattern noise, restrict to keywords appearing in ≥25 cohort films with usable rating distributions, then compute mean polar and mean rating per keyword. The pattern at the keyword level is robust to any single cluster's noise — it pools across the whole cohort.

In [ ]:
KW_STOP = {
    'one word title', 'two word title', 'three word title', 'four word title',
    'year in title', 'number in title', 'character name in title',
    'place name in title', 'directors name in title',
    '20th century', '21st century',
    '1900s', '1910s', '1920s', '1930s', '1940s', '1950s',
    '1960s', '1970s', '1980s', '1990s', '2000s', '2010s', '2020s',
}

kw_source = films.dropna(subset=['keywords']).copy()
kw_source['kw_list'] = kw_source['keywords'].str.split(',')
kw_long = (kw_source[['titleId', 'polar', 'mean_rating', 'love_share', 'hate_share', 'kw_list']]
           .explode('kw_list')
           .rename(columns={'kw_list': 'kw'}))
kw_long['kw'] = kw_long['kw'].astype(str).str.strip().str.lower()
kw_long = kw_long[~kw_long['kw'].isin(KW_STOP) & (kw_long['kw'] != '') & (kw_long['kw'] != 'nan')]

kw_stats = (kw_long.groupby('kw')
            .agg(n=('titleId', 'count'),
                 mean_polar=('polar', 'mean'),
                 mean_rating=('mean_rating', 'mean'),
                 mean_love=('love_share', 'mean'),
                 mean_hate=('hate_share', 'mean'))
            .query('n >= 25')
            .reset_index())

polar_med_kw  = kw_stats['mean_polar'].median()
rating_med_kw = kw_stats['mean_rating'].median()

def kw_quadrant(row):
    if row['mean_polar'] >= polar_med_kw and row['mean_rating'] >= rating_med_kw: return 'productive'
    if row['mean_polar'] >= polar_med_kw and row['mean_rating'] <  rating_med_kw: return 'rejected'
    if row['mean_polar'] <  polar_med_kw and row['mean_rating'] >= rating_med_kw: return 'consensus'
    return 'forgotten'
kw_stats['quadrant'] = kw_stats.apply(kw_quadrant, axis=1)

print(f'Keywords with ≥25 films and usable polar: {len(kw_stats):,}')
print(f'Median mean_polar across keywords:  {polar_med_kw:.3f}')
print(f'Median mean_rating across keywords: {rating_med_kw:.2f}')
display(Markdown('**Keyword counts per quadrant**'))
display(kw_stats['quadrant'].value_counts().to_frame('keywords'))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6.8))
qcolors = {'productive': PRODUCT, 'rejected': REJECT,
           'consensus':  PRIMARY,  'forgotten': GRAY}

for q, c in qcolors.items():
    sub = kw_stats[kw_stats['quadrant'] == q]
    ax.scatter(sub['mean_polar'], sub['mean_rating'],
               s=np.clip(sub['n'] / 2, 18, 240), alpha=0.55, color=c,
               edgecolor='none',
               label=f'{q.capitalize()}  ({len(sub)} kw)', zorder=2)

ax.set_xlim(kw_stats['mean_polar'].min() - 0.01, kw_stats['mean_polar'].max() + 0.02)
ax.set_ylim(kw_stats['mean_rating'].min() - 0.3, kw_stats['mean_rating'].max() + 0.3)
xmin, xmax = ax.get_xlim()
ymin, ymax = ax.get_ylim()

ax.add_patch(Rectangle((polar_med_kw, rating_med_kw), xmax-polar_med_kw, ymax-rating_med_kw,
                       facecolor=PRODUCT, alpha=0.07, zorder=0))
ax.add_patch(Rectangle((polar_med_kw, ymin), xmax-polar_med_kw, rating_med_kw-ymin,
                       facecolor=REJECT, alpha=0.07, zorder=0))
ax.axvline(polar_med_kw,  color='black', lw=0.7, ls='--', alpha=0.5)
ax.axhline(rating_med_kw, color='black', lw=0.7, ls='--', alpha=0.5)

# Label the most extreme keywords in productive + rejected quadrants
labels = pd.concat([
    kw_stats[kw_stats['quadrant'] == 'productive'].nlargest(8, 'mean_polar'),
    kw_stats[kw_stats['quadrant'] == 'rejected'].nlargest(8, 'mean_polar'),
])
for _, row in labels.iterrows():
    ax.annotate(row['kw'],
                xy=(row['mean_polar'], row['mean_rating']),
                xytext=(6, 4), textcoords='offset points',
                fontsize=8.5, color='#222')

ax.text(xmax - 0.005, ymax - 0.05, 'PRODUCTIVE LEXICON',
        ha='right', va='top', fontsize=11, fontweight='bold',
        color=PRODUCT, alpha=0.7)
ax.text(xmax - 0.005, ymin + 0.05, 'REJECTED LEXICON',
        ha='right', va='bottom', fontsize=11, fontweight='bold',
        color=REJECT, alpha=0.7)

ax.set_xlabel('Mean polarization of films tagged with this keyword')
ax.set_ylabel('Mean IMDb rating of films tagged with this keyword')
ax.set_title(f'The arthouse provocation lexicon · {len(kw_stats):,} keywords (≥25 films) · point size = film count',
             pad=12)
ax.legend(frameon=False, loc='lower left', fontsize=9.5)
for s in ('top','right'): ax.spines[s].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / '03_lexicon_scatter.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Top productive: above-median polar, ranked by rating (highest first)
# Rejected:      above-median polar, ranked by rating (lowest first)
prod_kws = (kw_stats[kw_stats['quadrant'] == 'productive']
            .nlargest(12, 'mean_rating'))
rej_kws  = (kw_stats[kw_stats['quadrant'] == 'rejected']
            .nsmallest(12, 'mean_rating'))

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.8))
x_max = max(prod_kws['mean_rating'].max(), rej_kws['mean_rating'].max()) + 1.4

for ax, df, color, title in [
    (axes[0], prod_kws.iloc[::-1], PRODUCT,
     'Productive provocation lexicon\n(above-median polar · top 12 by rating)'),
    (axes[1], rej_kws.iloc[::-1],  REJECT,
     'Rejected provocation lexicon\n(above-median polar · bottom 12 by rating)'),
]:
    ax.barh(df['kw'], df['mean_rating'], color=color, alpha=0.88)
    for i, (rating, n, polar) in enumerate(zip(df['mean_rating'], df['n'], df['mean_polar'])):
        ax.text(rating + 0.06, i, f'n={n} · polar={polar:.2f}',
                va='center', fontsize=8.5, color='gray')
    ax.set_xlabel('Mean IMDb rating of films tagged')
    ax.set_title(title, fontsize=11)
    ax.set_xlim(0, x_max)
    for s in ('top','right'): ax.spines[s].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / '04_lexicon_bars.png', dpi=200, bbox_inches='tight')
plt.show()

display(Markdown('**Productive provocation lexicon — full top 12**'))
display(prod_kws[['kw', 'n', 'mean_polar', 'mean_rating']]
        .round({'mean_polar': 3, 'mean_rating': 2})
        .reset_index(drop=True))
display(Markdown('**Rejected provocation lexicon — full bottom 12**'))
display(rej_kws[['kw', 'n', 'mean_polar', 'mean_rating']]
        .round({'mean_polar': 3, 'mean_rating': 2})
        .reset_index(drop=True))

## §5 — Recommendations for Publikum

**Headline finding.** The arthouse audience does not reward controversy uniformly. It rewards a *kind* of controversy. Films that provoke through formal experimentation — narrative ambiguity, essay-form, surrealism, slow cinema — sit in the productive quadrant: divisive *and* defended. Films that provoke through transgressive content — graphic sex, exploitation, gratuitous violence — sit in the rejected quadrant: divisive *and* dismissed. Both look identical on a single polarization score; they could not be more different on the rating axis.

**Three operational implications:**

1. **Positioning.** When a new film carries productive-lexicon keywords, market it as productively divisive — *the love-it-or-hate-it film of the year*. When it carries rejected-lexicon keywords, do not — the controversy will not convert. Use the polarization band as a marketing variable, not a single "is it controversial" flag.

2. **Comparable titles.** Comp matching should include a polarization-shape filter. A high-polar film with a J-shape distribution (love-heavy) is comp-compatible with another J-shape film. Pairing it with a U-shape or hate-heavy comp mis-sets stakeholder expectations and review variance.

3. **Greenlighting / acquisition.** The productive lexicon is a checklist. Projects whose keywords land in the upper-right quadrant of the lexicon scatter have, historically in this cohort, achieved both critical respect and engaged audiences. That is the bet an arthouse acquisition should be making — formal risk over content shock.

**Caveats kept honest.** Polarization is computed only on the ~32% of cohort films with ≥100 votes — small, recent, or non-Anglophone titles drop out. ROI is not in this analysis (n=91 in §7 of the general analysis is too thin). The lexicon is descriptive of past patterns, not a guarantee for new films. Quadrant medians shift if the cohort definition shifts.

## Pitch crib (~2 minutes)

**Slide 1 — `01_headline_scatter.png`**
> "Arthouse is supposed to polarize. We measured polarization for every cohort film with usable IMDb data — 1,352 films — and asked what the polarization score alone hides. Here's what we found: there isn't *one* kind of controversy. There are two." *[Point at green quadrant.]* "These films divide audiences and the audience defends them — Persona, Mulholland Drive, the canon. *[Point at red.]* These have the same polarization score on average — but the audience rejects them."

**Slide 2 (optional backup) — `02_mechanism_distributions.png`**
> "Same controversy score. Opposite distribution shape. A polarization score on its own is a trap."

**Slide 3 — `04_lexicon_bars.png`**
> "We pulled this apart at the keyword level so the recommendation is operational. *Experimental, avant-garde, essay, surrealism* — formal provocation — land in the productive lexicon. Words signaling transgressive content land in the rejected lexicon. The takeaway for Publikum: bet on form, not on shock."

**Closing line.**
> "Our recommendation is one variable, not a label. Don't ask *is this film controversial?* — ask *which kind of controversy does it carry?* Productive provocation is what arthouse audiences pay for."